# CodeSage: QLoRA Fine-Tuning on CommitPack (Qwen2.5-Coder-1.5B)

**Hardware:** Google Colab T4 / L4 / A100 GPU (Free or Pro)
**Base Model:** 
**Optimization:** Unsloth 4-bit QLoRA, rank 16, alpha 32, sequence length 1024
**Goal:** Train bug-fix repair adapter, measure CodeBLEU and HumanEval pass@1 before and after, generate empirical receipt.


## 1. Verify GPU Allocation
Make sure your runtime is set to **GPU**: *Runtime -> Change runtime type -> T4 GPU* (or A100).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Mount Google Drive (Optional)
Mounting Google Drive ensures checkpoints are automatically backed up and saved across disconnections.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3. Install Unsloth and ML Dependencies

In [ ]:
# Install Unsloth optimized kernels
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --no-deps trl peft accelerate bitsandbytes
!pip install "datasets==3.6.0" codebleu human-eval


## 4. Clone Repository & Checkout Branch

In [ ]:
# Option 1: Upload codesage-training.zip from your Desktop (No GitHub credentials needed!)
import os
from google.colab import files

if not os.path.exists("/content/Codesage/training"):
    print("Upload codesage-training.zip from your Desktop:")
    uploaded = files.upload()
    !mkdir -p /content/Codesage
    !unzip -o codesage-training.zip -d /content/Codesage/
    %cd /content/Codesage
else:
    %cd /content/Codesage
    print("Codesage training directory ready.")


## 5. Verify Colab Environment Constraints

In [ ]:
!python training/verify_colab_env.py


## 6. Dataset Preparation
Choose one option:
- **Option A (Fast dry-run, 10s):** Generate 160 seed samples to verify the training and export pipeline immediately.
- **Option B (Full production, 25-35m):** Stream 750,000 rows from  with 7 AST filters.

In [ ]:
# Run Option A for instant verification, or Option B for full dataset
!python training/seed_starter_data.py
# !python training/dataset_prep.py


## 7. Baseline Evaluation (Pre-Training)
Evaluate the zero-shot base  on held-out test data.

In [ ]:
!python training/eval_codebleu.py --model Qwen/Qwen2.5-Coder-1.5B-Instruct --test_data training/data/test.jsonl --output training/results/base_codeblu.json
!python training/eval_humaneval.py --model Qwen/Qwen2.5-Coder-1.5B-Instruct --output training/results/base_humaneval.json --temperature 0 --seed 42


## 8. Execute QLoRA Fine-Tuning
Trains 4-bit LoRA adapter across 3 epochs. Saves checkpoints to  and Google Drive.

In [ ]:
# Select model size: "7b" (Recommended, fits in ~10GB VRAM on T4) or "1.5b" (~7GB VRAM)
MODEL_SIZE = "7b"
!python training/finetune.py --model-size $MODEL_SIZE


## 9. Post-Training Evaluation
Evaluate fine-tuned checkpoint on the sacred held-out test set.

In [ ]:
!python training/eval_codebleu.py --model /content/checkpoints/best_checkpoint --test_data training/data/test.jsonl --output training/results/finetuned_codeblu.json
!python training/eval_humaneval.py --model /content/checkpoints/best_checkpoint --output training/results/finetuned_humaneval.json --temperature 0 --seed 42


## 10. Generate Comparison Receipt
Compares baseline vs fine-tuned CodeBLEU and HumanEval pass@1, generating .

In [ ]:
!python training/compare_results.py


## 11. Export LoRA Adapter & GGUF Quantization
Exports weights for deployment to vLLM, Ollama, or Hugging Face Hub.

In [ ]:
# Select model size: "7b" (Recommended, fits in ~10GB VRAM on T4) or "1.5b" (~7GB VRAM)
MODEL_SIZE = "7b"
!python training/finetune.py --model-size $MODEL_SIZE
